In [1]:
import torch
import numpy as np
from transformers import AutoTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments
from datasets import load_dataset, Dataset, DatasetDict
from tqdm import tqdm
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-base").to(device)

In [2]:
Nepoch = 30
batch_size = 2
learning_rate = 1e-5

In [3]:
def alternate_combine_n(_arr1, _arr2, n):
    combined_full = {}
    i = 0
    j = 0
    
    total_iterations = len(_arr1) // n + len(_arr2) // n
    for set in ['test', 'train', 'validation']:
        combined_set = {}
        for l in ['text','summary']:
            combined = []
            with tqdm(total=total_iterations, desc="Combining") as pbar:
                arr1 = _arr1[set][l]
                arr2 = _arr2[set][l]
                i = 0
                j = 0
                while i < len(arr1) and j < len(arr2):
                    for _ in range(n):
                        combined.append(arr1[i])
                        i += 1
                        if i >= len(arr1):
                            break
                    
                    pbar.update(1)
                    
                    for _ in range(n):
                        combined.append(arr2[j])
                        j += 1
                        if j >= len(arr2):
                            break
                    
                    pbar.update(1)
            
            combined.extend(arr1[i:])
            combined.extend(arr2[j:])
            combined_set[l] = combined
        combined_full[set] = combined_set
    return combined_full

In [4]:
n = batch_size * 2
dataset1 = load_dataset("TanveerAman/AMI-Corpus-Text-Summarization")
dataset2 = {'train':[],'validation':[],'test':[]}
num_of_data = [500,100,100]
for i, set in enumerate(dataset2.keys()):
    dataset2[set] = load_dataset("scientific_papers",'arxiv',split = f"{set}[:{num_of_data[i]}]")
for set in ['train','validation','test']:
    dataset1[set] = dataset1[set].rename_column('Dialogue','text').rename_column('Summaries','summary')
    dataset2[set] = dataset2[set].rename_column('article','text').rename_column('abstract','summary').rename_column('section_names','sections')



In [5]:
dataset_combined = alternate_combine_n(dataset1,dataset2,n)

Combining: 4it [00:00, 1104.42it/s]
Combining: 4it [00:00, 6181.73it/s]
Combining: 56it [00:00, 2381.46it/s]
Combining: 56it [00:00, 20982.76it/s]
Combining: 10it [00:00, 2501.08it/s]
Combining: 10it [00:00, 14883.97it/s]


In [6]:
dataset_combined = DatasetDict({'train':Dataset.from_dict(dataset_combined['train']),'validation':Dataset.from_dict(dataset_combined['validation']),'test':Dataset.from_dict(dataset_combined['test'])})

In [7]:
dataset_combined

DatasetDict({
    train: Dataset({
        features: ['text', 'summary'],
        num_rows: 610
    })
    validation: Dataset({
        features: ['text', 'summary'],
        num_rows: 120
    })
    test: Dataset({
        features: ['text', 'summary'],
        num_rows: 107
    })
})

In [8]:
def convert_to_features(example_batch):
    model_inputs = tokenizer(example_batch['text'],padding = True ,max_length=1024, truncation=True)
    labels = tokenizer(text_target=example_batch["summary"],padding = True , max_length=1024, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [9]:

dataset = dataset_combined.map(convert_to_features, batched=True)
columns = ['input_ids', 'labels', 'input_ids','attention_mask',] 
dataset.set_format(type='torch', columns=columns)

Map:   0%|          | 0/610 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/107 [00:00<?, ? examples/s]

In [10]:

training_args = TrainingArguments(
    output_dir='./models/bart-summarizer',          
    num_train_epochs=Nepoch,
    learning_rate=learning_rate,           
    per_device_train_batch_size=batch_size, 
    per_device_eval_batch_size=batch_size,   
    warmup_steps=100,               
    weight_decay=0.01,              
    logging_dir='./logs',
    save_steps=500,                  # number of updates steps before checkpoint saves
    save_total_limit=5,              # limit the total amount of checkpoints and deletes the older checkpoints
    evaluation_strategy='steps',     # evaluation strategy to adopt during training
    eval_steps=100,          
)

trainer = Trainer(
    model=model,
    tokenizer = tokenizer,                       
    args=training_args,                  
    train_dataset=dataset['train'],        
    eval_dataset=dataset['validation']   
)

In [ ]:
trainer.train()

/home/abdulna/miniconda3/envs/myenv2/lib/python3.9/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


wandb: Currently logged in as: aabdo20050. Use `wandb login --relogin` to force relogin


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You're using a BartTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss
100,No log,4.823753
200,No log,2.374504
300,No log,1.320947
400,No log,1.053596
500,3.651200,1.006968
600,3.651200,0.971606
700,3.651200,0.971427
800,3.651200,0.965615
900,3.651200,0.949237
1000,0.886400,0.943019
